# PyTorch cơ bản

Notebook này đi qua những phần cần nắm để bắt đầu với PyTorch:

1. Tensor: tạo, kiểu dữ liệu, shape
2. Phép toán và đổi shape
3. Tensor ↔ numpy ↔ ảnh OpenCV
4. CPU / GPU (device)
5. Autograd: tự tính đạo hàm
6. Tự train hồi quy tuyến tính bằng tay
7. Cách làm "chuẩn": `nn.Module`, loss, optimizer
8. Dataset và DataLoader
9. Bài tổng hợp: train mạng phân loại
10. Lưu và nạp model

Chạy lần lượt từng ô từ trên xuống (`Shift+Enter`).

## 0. Cài đặt

Nếu chưa cài PyTorch, bỏ dấu `#` ở dòng phù hợp rồi chạy ô dưới **một lần** (cài vào kernel `.venv` đang chọn):
- Bản CPU: nhẹ (~200–300MB), đủ để học.
- Bản GPU (CUDA): nặng (~2–3GB), dùng card NVIDIA.

In [ ]:
# %pip install torch --index-url https://download.pytorch.org/whl/cpu      # bản CPU
# %pip install torch --index-url https://download.pytorch.org/whl/cu126    # bản GPU (NVIDIA)

In [ ]:
import torch
import numpy as np

print("PyTorch version:", torch.__version__)
print("Có GPU không?  :", torch.cuda.is_available())

## 1. Tensor

**Tensor** là kiểu dữ liệu trung tâm của PyTorch: một mảng nhiều chiều, giống `numpy.ndarray`,
nhưng có thêm 2 khả năng:
- chạy trên **GPU**,
- **tự tính đạo hàm** (autograd), là thứ cần để train mạng nơ-ron.

| Số chiều | Tên gọi | Ví dụ |
|---|---|---|
| 0 | scalar | `3.14` |
| 1 | vector | `[1, 2, 3]` |
| 2 | ma trận | ảnh xám (H, W) |
| 3 | tensor 3D | ảnh màu (C, H, W) |
| 4 | tensor 4D | một batch ảnh (N, C, H, W) |

In [ ]:
# Tạo tensor từ list
a = torch.tensor([1, 2, 3])
b = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
print(a, a.shape, a.dtype)
print(b, b.shape, b.dtype)

In [ ]:
# Các hàm tạo tensor hay dùng
print(torch.zeros(2, 3))          # toàn 0
print(torch.ones(2, 3))           # toàn 1
print(torch.rand(2, 3))           # ngẫu nhiên đều trong [0, 1)
print(torch.randn(2, 3))          # ngẫu nhiên phân phối chuẩn (mean 0, std 1)
print(torch.arange(0, 10, 2))     # 0, 2, 4, 6, 8
print(torch.linspace(0, 1, 5))    # 5 số cách đều từ 0 đến 1

In [ ]:
# 3 thuộc tính quan trọng nhất của một tensor
x = torch.rand(3, 4)
print("shape :", x.shape)     # kích thước
print("dtype :", x.dtype)     # kiểu dữ liệu
print("device:", x.device)    # nằm ở CPU hay GPU

# Đổi kiểu dữ liệu
y = torch.tensor([1, 2, 3])            # int64
print(y.dtype, "->", y.float().dtype)  # float32 (mạng nơ-ron gần như luôn dùng float32)

## 2. Phép toán và đổi shape

Phép toán trên tensor được thực hiện **theo từng phần tử**, không cần vòng lặp `for`.

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
b = torch.tensor([10.0, 20.0, 30.0])

print(a + b)          # cộng từng phần tử
print(a * b)          # nhân từng phần tử
print(a ** 2)         # bình phương
print(a.sum(), a.mean(), a.max())

In [ ]:
# Nhân ma trận: dùng @ (hoặc torch.matmul), KHÁC với * (nhân từng phần tử)
A = torch.rand(2, 3)
B = torch.rand(3, 4)
C = A @ B
print(C.shape)        # (2, 3) @ (3, 4) -> (2, 4)

In [ ]:
# Broadcasting: tensor nhỏ tự "kéo giãn" để khớp tensor lớn
m = torch.ones(3, 4)
v = torch.tensor([1.0, 2.0, 3.0, 4.0])   # shape (4,)
print(m + v)                              # v được cộng vào từng hàng

In [ ]:
# Đổi shape
x = torch.arange(12)
print(x.shape)                 # (12,)
print(x.reshape(3, 4))         # thành ma trận 3x4
print(x.reshape(2, -1).shape)  # -1 = để PyTorch tự tính -> (2, 6)

img = torch.rand(3, 32, 32)            # 1 ảnh (C, H, W)
batch = img.unsqueeze(0)               # thêm chiều batch ở đầu
print(batch.shape)                     # (1, 3, 32, 32)
print(batch.squeeze(0).shape)          # bỏ chiều có kích thước 1 -> (3, 32, 32)
print(img.permute(1, 2, 0).shape)      # đổi thứ tự chiều -> (32, 32, 3)

In [ ]:
# Indexing / slicing giống numpy
x = torch.arange(12).reshape(3, 4)
print(x)
print(x[0])          # hàng đầu tiên
print(x[:, 1])       # cột thứ 2
print(x[1:, 2:])     # cắt một vùng
print(x[x > 5])      # lọc theo điều kiện

## 3. Tensor ↔ numpy ↔ ảnh OpenCV

Ảnh OpenCV là mảng numpy dạng **(H, W, C)**, màu **BGR**, kiểu `uint8` (0–255).
Mạng nơ-ron trong PyTorch lại muốn ảnh dạng **(N, C, H, W)**, kiểu `float32`, thường chuẩn hoá về 0–1.

In [ ]:
# numpy <-> tensor
arr = np.array([[1, 2], [3, 4]])
t = torch.from_numpy(arr)      # numpy -> tensor (dùng chung bộ nhớ)
back = t.numpy()               # tensor -> numpy
print(t, type(back))

In [ ]:
import cv2

img = cv2.imread("person.jpg")                    # (H, W, 3), BGR, uint8
img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)        # đổi sang RGB
print("numpy :", img.shape, img.dtype)

x = torch.from_numpy(img).permute(2, 0, 1)        # (H, W, C) -> (C, H, W)
x = x.float() / 255.0                             # uint8 0..255 -> float 0..1
x = x.unsqueeze(0)                                # thêm chiều batch -> (1, C, H, W)
print("tensor:", x.shape, x.dtype, x.min().item(), x.max().item())

## 4. CPU và GPU

Tensor và model phải nằm **cùng một device** thì mới tính toán với nhau được.
Cách viết dưới đây chạy được cả khi có và không có GPU.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Đang dùng:", device)

x = torch.rand(3, 3).to(device)      # chuyển tensor sang device
print(x.device)

# Muốn chuyển về numpy phải đưa về CPU trước
print(x.cpu().numpy())

## 5. Autograd: tự tính đạo hàm

Đây là "phép màu" giúp train mạng nơ-ron. Đánh dấu `requires_grad=True`, làm phép tính,
gọi `.backward()` là PyTorch tự tính đạo hàm.

Ví dụ: $y = x^2 + 3x$, đạo hàm $\frac{dy}{dx} = 2x + 3$. Tại $x = 2$ thì đạo hàm bằng $7$.

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x ** 2 + 3 * x

y.backward()          # tính dy/dx
print("y     =", y.item())
print("dy/dx =", x.grad.item())   # 7.0

In [ ]:
# Với nhiều biến: PyTorch tính đạo hàm riêng theo từng biến
w = torch.tensor(1.0, requires_grad=True)
b = torch.tensor(0.5, requires_grad=True)
x = torch.tensor(3.0)

y = w * x + b
y.backward()
print("dy/dw =", w.grad.item())   # = x = 3
print("dy/db =", b.grad.item())   # = 1

# Khi chỉ dự đoán (không train), tắt autograd cho nhanh và nhẹ
with torch.no_grad():
    y2 = w * x + b
print(y2.requires_grad)           # False

## 6. Train hồi quy tuyến tính "bằng tay"

Cho dữ liệu sinh ra từ $y = 2x + 1$ (có thêm nhiễu). Mục tiêu: để máy **tự tìm ra** $w \approx 2$ và $b \approx 1$.

Vòng train gồm 4 bước, lặp lại nhiều lần:
1. **Dự đoán**: `y_pred = w * x + b`
2. **Tính loss**: dự đoán sai bao nhiêu (ở đây dùng sai số bình phương trung bình, MSE)
3. **Backward**: tính đạo hàm của loss theo `w`, `b`
4. **Cập nhật**: dịch `w`, `b` ngược hướng đạo hàm một bước nhỏ (gradient descent)

In [ ]:
torch.manual_seed(0)                       # cố định ngẫu nhiên để kết quả lặp lại được
X = torch.rand(100, 1) * 10                # 100 điểm x trong [0, 10)
Y = 2 * X + 1 + torch.randn(100, 1)        # y = 2x + 1 + nhiễu

w = torch.randn(1, requires_grad=True)
b = torch.zeros(1, requires_grad=True)
lr = 0.01                                  # learning rate: độ lớn mỗi bước

for epoch in range(2000):
    Y_pred = w * X + b                     # 1. dự đoán
    loss = ((Y_pred - Y) ** 2).mean()      # 2. loss (MSE)
    loss.backward()                        # 3. tính đạo hàm

    with torch.no_grad():                  # 4. cập nhật (không cần theo dõi đạo hàm ở bước này)
        w -= lr * w.grad
        b -= lr * b.grad
    w.grad.zero_()                         # QUAN TRỌNG: xoá đạo hàm cũ, nếu không nó sẽ cộng dồn
    b.grad.zero_()

    if epoch % 400 == 0:
        print(f"epoch {epoch:4d} | loss {loss.item():.4f} | w {w.item():.3f} | b {b.item():.3f}")

print(f"Kết quả: w = {w.item():.3f} (đúng là 2), b = {b.item():.3f} (đúng là 1)")

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(X, Y, s=10, label="dữ liệu")
xs = torch.linspace(0, 10, 50).unsqueeze(1)
with torch.no_grad():
    plt.plot(xs, w * xs + b, "r", label="đường máy học được")
plt.legend(); plt.show()

## 7. Cách làm "chuẩn": `nn.Module`, loss, optimizer

Thực tế không ai tự viết `w`, `b` và tự cập nhật. PyTorch có sẵn:
- `torch.nn`: các lớp mạng (`nn.Linear`, `nn.Conv2d`, `nn.ReLU`...) và hàm loss (`nn.MSELoss`, `nn.CrossEntropyLoss`...)
- `torch.optim`: thuật toán cập nhật (`SGD`, `Adam`...)

Bài trên viết lại theo cách chuẩn:

In [ ]:
import torch.nn as nn

model = nn.Linear(in_features=1, out_features=1)       # chính là y = w*x + b
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.01)

for epoch in range(2000):
    Y_pred = model(X)                  # 1. dự đoán
    loss = loss_fn(Y_pred, Y)          # 2. loss
    optimizer.zero_grad()              #    xoá đạo hàm cũ
    loss.backward()                    # 3. backward
    optimizer.step()                   # 4. cập nhật

print("w =", model.weight.item(), "| b =", model.bias.item())

**Vòng train này là khung chung cho mọi model trong PyTorch**, từ hồi quy đơn giản đến YOLO:

```python
for epoch in range(...):
    pred = model(x)
    loss = loss_fn(pred, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
```

### Tự định nghĩa model
Model phức tạp hơn thì viết thành class kế thừa `nn.Module`:
- `__init__`: khai báo các lớp
- `forward`: dữ liệu đi qua các lớp thế nào

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden, out_dim)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

net = MLP(in_dim=2, hidden=16, out_dim=3)
print(net)
print("Số tham số:", sum(p.numel() for p in net.parameters()))

out = net(torch.rand(5, 2))      # batch 5 mẫu, mỗi mẫu 2 đặc trưng
print(out.shape)                 # (5, 3)

## 8. Dataset và DataLoader

Dữ liệu thật thường lớn, không đưa hết vào model một lần được. `DataLoader` giúp:
- chia dữ liệu thành từng **batch** nhỏ,
- **xáo trộn** (shuffle) mỗi epoch.

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

dataset = TensorDataset(X, Y)                       # ghép cặp (x, y)
loader = DataLoader(dataset, batch_size=16, shuffle=True)

print("Số mẫu:", len(dataset), "| số batch:", len(loader))
xb, yb = next(iter(loader))                         # lấy thử 1 batch
print(xb.shape, yb.shape)                           # (16, 1) (16, 1)

## 9. Bài tổng hợp: phân loại điểm 2D thành 3 lớp

Gộp tất cả những gì ở trên: tạo dữ liệu, `DataLoader`, model `MLP`, `CrossEntropyLoss`, `Adam`, chạy trên GPU nếu có,
tách tập train/test và đo độ chính xác.

Dữ liệu: 3 đám điểm quanh 3 tâm khác nhau, mỗi đám là một lớp.

In [ ]:
torch.manual_seed(42)
centers = torch.tensor([[0.0, 0.0], [4.0, 4.0], [0.0, 5.0]])
N = 300                                              # số điểm mỗi lớp
X_all = torch.cat([c + torch.randn(N, 2) for c in centers])
y_all = torch.cat([torch.full((N,), i) for i in range(3)])   # nhãn 0, 1, 2

# Xáo trộn rồi chia 80% train / 20% test
perm = torch.randperm(len(X_all))
X_all, y_all = X_all[perm], y_all[perm]
split = int(0.8 * len(X_all))
train_loader = DataLoader(TensorDataset(X_all[:split], y_all[:split]), batch_size=32, shuffle=True)
X_test, y_test = X_all[split:], y_all[split:]

plt.scatter(X_all[:, 0], X_all[:, 1], c=y_all, cmap="viridis", s=8); plt.title("Dữ liệu"); plt.show()

In [ ]:
model = MLP(in_dim=2, hidden=16, out_dim=3).to(device)
loss_fn = nn.CrossEntropyLoss()                      # loss cho bài phân loại
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

for epoch in range(20):
    model.train()                                    # chế độ train
    total = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)                           # (batch, 3): điểm số cho từng lớp
        loss = loss_fn(logits, yb)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total += loss.item()
    if epoch % 5 == 0 or epoch == 19:
        print(f"epoch {epoch:2d} | loss trung bình {total / len(train_loader):.4f}")

In [ ]:
model.eval()                                         # chế độ đánh giá
with torch.no_grad():
    logits = model(X_test.to(device))
    pred = logits.argmax(dim=1).cpu()                # lớp có điểm cao nhất
acc = (pred == y_test).float().mean().item()
print(f"Độ chính xác trên tập test: {acc * 100:.1f}%")

## 10. Lưu và nạp model

Cách khuyên dùng: chỉ lưu **trọng số** (`state_dict`), khi nạp thì tạo lại model cùng cấu trúc rồi đổ trọng số vào.

In [ ]:
torch.save(model.state_dict(), "mlp.pth")

model2 = MLP(in_dim=2, hidden=16, out_dim=3)
model2.load_state_dict(torch.load("mlp.pth", map_location="cpu"))
model2.eval()

with torch.no_grad():
    print("Điểm (4, 4) thuộc lớp:", model2(torch.tensor([[4.0, 4.0]])).argmax(dim=1).item())   # mong đợi lớp 1

## Tóm tắt

| Khái niệm | Ghi nhớ |
|---|---|
| Tensor | mảng nhiều chiều, có `shape`, `dtype`, `device` |
| Ảnh | OpenCV `(H, W, C)` uint8 → PyTorch `(N, C, H, W)` float 0–1 |
| Device | `.to(device)`; model và dữ liệu phải cùng device |
| Autograd | `requires_grad=True` → `loss.backward()` → `.grad` |
| Vòng train | `pred → loss → zero_grad → backward → step` |
| Model | class kế thừa `nn.Module`, có `__init__` và `forward` |
| Dữ liệu | `Dataset` + `DataLoader` để chia batch, xáo trộn |
| Đánh giá | `model.eval()` + `torch.no_grad()` |

**Học tiếp:** mạng tích chập (`nn.Conv2d`) cho ảnh, `torchvision` (dataset MNIST/CIFAR, model có sẵn), rồi đến YOLO cho detection/pose.